# Oelen 2022 — SA + CA Results Inspector

Inspect the results of `train_oelen_stage2sa_stage3ca.py` on the Oelen et al. 2022
(1M-scBloodNL, v3) pathogen dataset.

**4 conditions:**
- `UT` — unstimulated control (CONTROL)
- `24hCA` — *C. albicans* 24 h (MED — intermediate cascade)
- `24hMTB` — *M. tuberculosis* 24 h (HARD — most cascade-dependent)
- `24hPA` — *P. aeruginosa* 24 h (EASY — direct TLR4/TLR5 activation)

**Expected learnability ordering (pre-registered biological ground truth):**
> 24hPA (easiest) > 24hCA (intermediate) > 24hMTB (hardest)

**No train/val split** — all donors used for training (learnability validation, not generalisation test).

---

## Table of Contents
1. Condition Groups & Difficulty Map
2. Run Log
3. Saved Figures
4. Load Dynamics
5. Stage 2 (SA) Learnability Ranking
6. Stage 3 (CA-only) Loss and CA Norm Trajectories
7. Stage 3 (CA-only) Learnability Ranking
8. Stage 2 vs Stage 3 Rank Comparison
9. Confusion Entropy Ranking
10. SA vs CA Attention Entropy per Condition
11. Cell-type Confidence Trajectories
12. Normalized AUC vs P_max Scatter

In [ ]:
import json
import pickle
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from pathlib import Path
from IPython.display import Image, display, Markdown
from scipy.stats import spearmanr
import anndata

from cytokine_mil.analysis.dynamics import (
    aggregate_to_donor_level,
    rank_cytokines_by_learnability,
    compute_confusion_entropy_summary,
)

In [ ]:
RESULTS_BASE = Path("/cs/labs/mornitzan/yam.arieli/results/oelen_sa_ca")

# List available runs
available = sorted(RESULTS_BASE.glob("run_*"))
print(f"Available runs under {RESULTS_BASE}:")
for p in available:
    print(f"  {p.name}")

In [ ]:
# Set to the run you want to inspect (default: latest)
run_dir = available[-1]
print(f"Inspecting: {run_dir}")

def _load_run(run_dir):
    """Load all standard artifacts from a run directory."""
    artifacts = {}
    for fname in ["dynamics_stage2.pkl", "dynamics_stage3.pkl", "label_encoder.json"]:
        p = run_dir / fname
        if p.exists():
            artifacts[fname] = p
    return artifacts

artifacts = _load_run(run_dir)
print(f"Found artifacts: {list(artifacts.keys())}")

## 1. Condition Groups & Difficulty Map

In [ ]:
DIFFICULTY_MAP = {
    "24hPA":  "EASY",
    "24hCA":  "MED",
    "24hMTB": "HARD",
    "UT":     "CONTROL",
}

# Display order: easiest → hardest, control last
ORDERED_CONDITIONS = ["24hPA", "24hCA", "24hMTB", "UT"]

COLOR_MAP = {
    "24hPA":  "steelblue",
    "24hCA":  "darkorange",
    "24hMTB": "tomato",
    "UT":     "gray",
}

display(Markdown(
    "| Condition | Difficulty | Mechanism |\n"
    "|-----------|------------|-----------|\n"
    "| 24hPA     | EASY       | Direct TLR4/TLR5 activation across multiple cell types — no cascade required |\n"
    "| 24hCA     | MED        | Intermediate cascade — monocyte → NK/T cell relay |\n"
    "| 24hMTB    | HARD       | Most cascade-dependent: phagocytosis → IL-12 → IFN-γ → macrophage reprogramming |\n"
    "| UT        | CONTROL    | Unstimulated baseline |\n"
))

## 2. Run Log

In [ ]:
log_path = run_dir / "run_log.txt"
if log_path.exists():
    print(log_path.read_text())
else:
    print("run_log.txt not found.")

## 3. Saved Figures

### 3.1 Stage 2 (SA) Learning Curves

In [ ]:
s2_pngs = sorted(run_dir.glob("learning_curves_stage_2*")) + \
          sorted(run_dir.glob("learning_curves_stage2*"))
for p in s2_pngs:
    display(Image(str(p)))

### 3.2 Stage 3 (CA-only) Learning Curves

In [ ]:
s3_pngs = sorted(run_dir.glob("learning_curves_stage_3*")) + \
          sorted(run_dir.glob("learning_curves_stage3*"))
for p in s3_pngs:
    display(Image(str(p)))

### 3.3 SA vs CA Attention Entropy

In [ ]:
for p in sorted(run_dir.glob("sa_vs_ca_entropy_*.png")):
    display(Image(str(p)))

### 3.4 CA Weight Norm Trajectory

In [ ]:
for p in sorted(run_dir.glob("ca_weight_norm_*.png")):
    display(Image(str(p)))

## 4. Load Dynamics

In [ ]:
with open(run_dir / "dynamics_stage2.pkl", "rb") as fh:
    dyn2 = pickle.load(fh)

with open(run_dir / "dynamics_stage3.pkl", "rb") as fh:
    dyn3 = pickle.load(fh)

records_s2     = dyn2["records"]
logged_s2      = dyn2["logged_epochs"]
records_s3     = dyn3["records"]
logged_s3      = dyn3["logged_epochs"]
ca_norm_traj   = dyn3["ca_weight_norm_trajectory"]
loss_traj      = dyn3["loss_trajectory"]
confusion_traj = dyn3["confusion_entropy_trajectory"]

print(f"Stage 2 — logged epochs: {len(logged_s2)}, records: {len(records_s2)}")
print(f"Stage 3 — logged epochs: {len(logged_s3)}, records: {len(records_s3)}")
print(f"Conditions in Stage 3 records: {sorted({r['cytokine'] for r in records_s3})}")

# Sanity check: SA entropy variance should be ~0 (frozen SA)
if records_s3:
    # Find a record with entropy_trajectory
    sample = next((r for r in records_s3 if r.get("entropy_trajectory")), None)
    if sample and len(sample["entropy_trajectory"]) > 1:
        sa_var = float(np.var(sample["entropy_trajectory"]))
        status = "OK" if sa_var < 1e-6 else "WARNING: SA may not be frozen!"
        print(f"SA entropy variance (first record): {sa_var:.2e}  [{status}]")

## 5. Stage 2 (SA) Learnability Ranking

In [ ]:
donor_traj_s2 = aggregate_to_donor_level(records_s2)
learn_s2      = rank_cytokines_by_learnability(donor_traj_s2, exclude=[])
ranking_s2    = learn_s2["ranking"]

print(f"Condition learnability ranking — Stage 2 (SA)")
print(f"Metric: {learn_s2['metric_description']}")
print()
print(f"{'Rank':>4}  {'Condition':<12}  {'Train AUC':>9}  Difficulty")
print("-" * 46)
for i, (cond, auc) in enumerate(ranking_s2, 1):
    difficulty = DIFFICULTY_MAP.get(cond, "")
    print(f"  {i:2d}.  {cond:<12}  {auc:>9.3f}  {difficulty}")

## 6. Stage 3 (CA-only) Loss and CA Norm Trajectories

In [ ]:
epochs_all = list(range(1, len(loss_traj) + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: training loss
axes[0].plot(epochs_all, loss_traj, color="steelblue", lw=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Mean cross-entropy loss")
axes[0].set_title("Stage 3 CA-only — Training loss")
axes[0].grid(True, alpha=0.3)

# Right: CA weight norm
init_norm  = ca_norm_traj[0]
final_norm = ca_norm_traj[-1]
delta_norm = final_norm - init_norm
axes[1].plot(epochs_all, ca_norm_traj, color="darkviolet", lw=2)
axes[1].axhline(init_norm, color="gray", ls="--", lw=0.8, label=f"Initial = {init_norm:.4f}")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("L2 norm of CA parameters")
axes[1].set_title(
    f"CA weight norm  |  init={init_norm:.4f}  final={final_norm:.4f}  Δ={delta_norm:+.4f}"
)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle(
    "Stage 3 CA-only diagnostics — Oelen 2022 pathogen\n"
    "Metric: L2 norm of [V_ca.weight, V_ca.bias, w_ca.weight, U_ca.weight] concatenated",
    fontsize=9,
)
plt.tight_layout()
plt.show()

## 7. Stage 3 (CA-only) Learnability Ranking

In [ ]:
donor_traj_s3 = aggregate_to_donor_level(records_s3)
learn_s3      = rank_cytokines_by_learnability(donor_traj_s3, exclude=[])
ranking_s3    = learn_s3["ranking"]

print(f"Condition learnability ranking — Stage 3 CA-only")
print(f"Metric: {learn_s3['metric_description']}")
print()
print(f"{'Rank':>4}  {'Condition':<12}  {'Train AUC':>9}  Difficulty")
print("-" * 46)
for i, (cond, auc) in enumerate(ranking_s3, 1):
    difficulty = DIFFICULTY_MAP.get(cond, "")
    print(f"  {i:2d}.  {cond:<12}  {auc:>9.3f}  {difficulty}")

# Biological validation
auc_map = {c: a for c, a in ranking_s3}
pa_auc  = auc_map.get("24hPA",  float("nan"))
ca_auc  = auc_map.get("24hCA",  float("nan"))
mtb_auc = auc_map.get("24hMTB", float("nan"))
print()
print("Biological ordering validation (pre-registered):")
print(f"  PA > CA:  {pa_auc > ca_auc}   (expected: True)")
print(f"  CA > MTB: {ca_auc > mtb_auc}  (expected: True)")
print(f"  PA > MTB: {pa_auc > mtb_auc}  (expected: True)")

## 8. Stage 2 vs Stage 3 Rank Comparison

In [ ]:
s2_order = [c for c, _ in ranking_s2]
s3_order = [c for c, _ in ranking_s3]
s2_auc   = {c: a for c, a in ranking_s2}
s3_auc   = {c: a for c, a in ranking_s3}

if set(s2_order) == set(s3_order) and len(s2_order) >= 2:
    s3_rank = {c: i for i, c in enumerate(s3_order)}
    s3_aligned = [s3_rank.get(c, len(s3_order)) for c in s2_order]
    rho, pval  = spearmanr(range(len(s2_order)), s3_aligned)
    print(f"Stage 2 vs Stage 3 rank correlation: Spearman ρ = {rho:.3f}  (p = {pval:.3f})")
    print(f"Stable (ρ > 0.7): {rho > 0.7}")
    print()

print(f"{'Condition':<12}  {'S2 AUC':>8}  {'S2 Rank':>7}  {'S3 AUC':>8}  {'S3 Rank':>7}  "
      f"{'ΔAUC':>7}  {'ΔRank':>6}  Difficulty")
print("-" * 75)
for i, cond in enumerate(s2_order, 1):
    s3r = s3_rank.get(cond, "?") + 1 if isinstance(s3_rank.get(cond), int) else "?"
    da  = s3_auc.get(cond, float("nan")) - s2_auc.get(cond, float("nan"))
    dr  = (s3_rank.get(cond, i - 1) + 1) - i
    arrow = "↑" if dr < 0 else ("↓" if dr > 0 else "=")
    difficulty = DIFFICULTY_MAP.get(cond, "")
    print(
        f"  {cond:<12}  {s2_auc.get(cond, float('nan')):>8.3f}  {i:>7d}  "
        f"{s3_auc.get(cond, float('nan')):>8.3f}  {s3r:>7}  "
        f"{da:>+7.3f}  {dr:>5d}{arrow}  {difficulty}"
    )

## 9. Confusion Entropy Ranking

In [ ]:
conf_result = compute_confusion_entropy_summary(confusion_traj, exclude=[])

print("Condition confusion entropy ranking — Stage 3 CA-only")
print(f"Metric: {conf_result['metric_description']}")
print()
print(f"{'Condition':<12}  {'Train AUC(H_c)':>14}  Difficulty")
print("-" * 38)
for cond, auc in conf_result["ranking"]:
    difficulty = DIFFICULTY_MAP.get(cond, "")
    print(f"  {cond:<12}  {auc:>14.3f}  {difficulty}")

print()
print("Interpretation: high H_confusion → model is confused across many classes (genuinely hard).")
print("Low H_confusion → confusion concentrated on specific similar conditions.")

## 10. SA vs CA Attention Entropy per Condition

In [ ]:
def _extract_layer_entropy(records, cytokine):
    """Return SA and CA entropy curves (one array per tube) for a condition."""
    sa_curves, ca_curves = [], []
    for rec in records:
        if rec["cytokine"] != cytokine:
            continue
        sa_traj = rec.get("entropy_trajectory")
        ca_traj = rec.get("entropy_trajectory_ca")
        if sa_traj is None or ca_traj is None:
            continue
        sa_curves.append(np.array(sa_traj))
        ca_curves.append(np.array(ca_traj))
    return sa_curves, ca_curves


n_cond = len(ORDERED_CONDITIONS)
fig, axes = plt.subplots(1, n_cond, figsize=(5 * n_cond, 4), sharey=True)

for ax, cond in zip(axes, ORDERED_CONDITIONS):
    difficulty = DIFFICULTY_MAP.get(cond, "")
    sa_curves, ca_curves = _extract_layer_entropy(records_s3, cond)
    mean_sa = mean_ca = None
    if sa_curves:
        mean_sa = np.mean(sa_curves, axis=0)
        mean_ca = np.mean(ca_curves, axis=0)
        ax.plot(logged_s3, mean_sa, color="steelblue", linewidth=2, label="SA (frozen)")
        ax.plot(logged_s3, mean_ca, color="darkorange", linewidth=2,
                linestyle="--", label="CA (trainable)")
    sa_delta = float(np.ptp(mean_sa)) if mean_sa is not None else 0.0
    ax.set_title(f"{cond}\n({difficulty})  SA Δ={sa_delta:.4f}", fontsize=9)
    ax.set_xlabel("Epoch (Stage 3)")
    ax.set_ylabel("H(a) [nats]")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    "Stage 3 CA-only — SA vs CA attention entropy per condition  |  Oelen 2022\n"
    "Metric: H(a) = -sum_i a_i log(a_i). SA is frozen (should be flat, Δ ≈ 0);\n"
    "CA changes only if it learned signal. Mean across pseudo-tubes per condition.",
    fontsize=9,
)
plt.tight_layout(rect=[0, 0, 1, 0.88])
plt.show()

## 11. Cell-type Confidence Trajectories

Cell type labels loaded from the pseudo-tube h5ad files via `cell_type_lowerres`.

In [ ]:
# Load cell-type labels from pseudo-tube h5ad files
# Each record has a 'tube_path' pointing to the .h5ad file
cell_type_obs = {}  # tube_path -> np.array of cell-type strings

unique_paths = sorted({r["tube_path"] for r in records_s3})
print(f"Loading cell-type labels from {len(unique_paths)} pseudo-tube files...")

for path in unique_paths:
    try:
        ad = anndata.read_h5ad(path, backed="r")
        # Oelen dataset uses cell_type_lowerres for coarse cell type
        if "cell_type_lowerres" in ad.obs.columns:
            cell_type_obs[path] = ad.obs["cell_type_lowerres"].values
        elif "cell_type" in ad.obs.columns:
            cell_type_obs[path] = ad.obs["cell_type"].values
        else:
            cell_type_obs[path] = np.array(["unknown"] * ad.n_obs)
        ad.file.close()
    except Exception as e:
        print(f"  WARNING: could not load {path}: {e}")

all_cell_types = sorted({ct for cts in cell_type_obs.values() for ct in cts})
print(f"Cell types found ({len(all_cell_types)}): {all_cell_types}")

In [ ]:
def _compute_ct_trajectories(records, cell_type_obs, cytokine, conf_key):
    """
    Compute mean confidence trajectory per cell type for a given condition.

    Aggregation: median across pseudo-tubes per donor, then mean across donors.
    Returns: {cell_type: np.array(n_logged_epochs)}
    """
    # group by donor
    donor_ct = defaultdict(lambda: defaultdict(list))  # donor -> ct -> list of trajectories
    for rec in records:
        if rec["cytokine"] != cytokine:
            continue
        traj = rec.get(conf_key)
        if traj is None:
            continue
        path = rec["tube_path"]
        cts  = cell_type_obs.get(path)
        if cts is None:
            continue
        n_cells, n_epochs = traj.shape
        for ct in np.unique(cts):
            mask = cts == ct
            mean_traj = traj[mask].mean(axis=0)  # (n_epochs,)
            donor_ct[rec["donor"]][ct].append(mean_traj)

    # median per donor, then mean across donors
    ct_trajectories = defaultdict(list)
    for donor, ct_dict in donor_ct.items():
        for ct, trajs in ct_dict.items():
            ct_trajectories[ct].append(np.median(trajs, axis=0))

    return {ct: np.mean(trajs, axis=0) for ct, trajs in ct_trajectories.items()}


def _plot_ct_sa_vs_ca(records, cell_type_obs, cytokine, logged_epochs, ax_sa, ax_ca):
    """Plot SA vs CA cell-type confidence trajectories on provided axes."""
    sa_ct = _compute_ct_trajectories(records, cell_type_obs, cytokine, "confidence_trajectory_sa")
    ca_ct = _compute_ct_trajectories(records, cell_type_obs, cytokine, "confidence_trajectory_ca")

    all_cts = sorted(set(sa_ct) | set(ca_ct))
    cmap    = plt.cm.tab20.colors

    for ci, ct in enumerate(all_cts):
        color = cmap[ci % len(cmap)]
        if ct in sa_ct:
            ax_sa.plot(logged_epochs, sa_ct[ct], color=color, label=ct, lw=1.5)
        if ct in ca_ct:
            ax_ca.plot(logged_epochs, ca_ct[ct], color=color, label=ct, lw=1.5)

    difficulty = DIFFICULTY_MAP.get(cytokine, "")
    ax_sa.set_title(f"{cytokine} ({difficulty}) — SA layer (frozen)", fontsize=9)
    ax_ca.set_title(f"{cytokine} ({difficulty}) — CA layer (trainable)", fontsize=9)
    for ax in [ax_sa, ax_ca]:
        ax.set_xlabel("Epoch (Stage 3)")
        ax.set_ylabel("Mean C_i(t) = a_i(t) · P(Y_correct)")
        ax.legend(fontsize=6, ncol=2)
        ax.grid(True, alpha=0.3)

    return sa_ct, ca_ct

### 11.1 SA vs CA per condition

In [ ]:
ct_results = {}  # cytokine -> {"sa": ..., "ca": ...}

for cond in ORDERED_CONDITIONS:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
    sa_ct, ca_ct = _plot_ct_sa_vs_ca(
        records_s3, cell_type_obs, cond, logged_s3, axes[0], axes[1]
    )
    ct_results[cond] = {"sa": sa_ct, "ca": ca_ct}

    fig.suptitle(
        f"{cond} — cell-type confidence trajectories  |  Oelen 2022\n"
        "Metric: AUC(mean C_i(t) = a_i(t) · P(Y_correct)), "
        "aggregated to donor level (median per donor, mean across donors)",
        fontsize=9,
    )
    plt.tight_layout()
    plt.show()

### 11.2 Cell-type Activation Heatmaps

In [ ]:
ACTIVATION_THRESHOLD = 0.05

def _activation_epoch(traj, threshold=ACTIVATION_THRESHOLD):
    """First logged epoch index where mean C_i(t) crosses threshold. NaN if never."""
    for ei, v in enumerate(traj):
        if v >= threshold:
            return ei
    return float("nan")


def _plot_activation_heatmap(ct_results_by_cond, logged_epochs, layer_key, title):
    """Heatmap: rows = conditions (ordered by Stage 3 AUC), cols = cell types."""
    all_cts = sorted({ct for d in ct_results_by_cond.values() for ct in d.get(layer_key, {})})
    cond_order = [c for c, _ in ranking_s3]  # ordered by Stage 3 learnability

    matrix = np.full((len(cond_order), len(all_cts)), float("nan"))
    for ri, cond in enumerate(cond_order):
        ct_traj = ct_results_by_cond.get(cond, {}).get(layer_key, {})
        for ci, ct in enumerate(all_cts):
            if ct in ct_traj:
                matrix[ri, ci] = _activation_epoch(ct_traj[ct])

    fig, ax = plt.subplots(figsize=(max(10, len(all_cts) * 0.8), max(4, len(cond_order) * 0.6)))
    im = ax.imshow(matrix, aspect="auto", cmap="YlOrRd_r", interpolation="none")
    plt.colorbar(im, ax=ax, label="First logged epoch crossing threshold")
    ax.set_xticks(range(len(all_cts)))
    ax.set_xticklabels(all_cts, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(cond_order)))
    ax.set_yticklabels(
        [f"{c}  ({DIFFICULTY_MAP.get(c, '')})" for c in cond_order], fontsize=8
    )
    ax.set_xlabel("Cell type (cell_type_lowerres)")
    ax.set_ylabel("Condition (ordered by Stage 3 learnability AUC ↓)")
    ax.set_title(
        f"{title}\n"
        f"Threshold = {ACTIVATION_THRESHOLD}  |  NaN (white) = never crossed threshold"
    )
    plt.tight_layout()
    plt.show()


_plot_activation_heatmap(
    ct_results, logged_s3, "sa",
    "Cell-type activation epoch — SA layer (frozen, Stage 2 representation)\n"
    "Metric: first logged epoch where mean C_i(t) = a_SA_i(t) · P(Y_correct) >= 0.05",
)

_plot_activation_heatmap(
    ct_results, logged_s3, "ca",
    "Cell-type activation epoch — CA layer (trainable, Stage 3)\n"
    "Metric: first logged epoch where mean C_i(t) = a_CA_i(t) · P(Y_correct) >= 0.05",
)

### 11.3 Cascade Ordering — SA-only vs CA-only vs Hub Cell Types

In [ ]:
def _classify_cell_types(sa_ct, ca_ct, threshold=ACTIVATION_THRESHOLD):
    """
    Classify cell types per condition into:
      SA-only  — activated in SA but not CA (direct targets of the perturbation)
      CA-only  — activated in CA but not SA (cascade / indirect responders)
      Both     — hub cells (activate in both layers)
    """
    def _crossed(traj):
        return any(v >= threshold for v in traj)

    sa_active = {ct for ct, traj in sa_ct.items() if _crossed(traj)}
    ca_active = {ct for ct, traj in ca_ct.items() if _crossed(traj)}

    return {
        "SA-only":  sorted(sa_active - ca_active),
        "CA-only":  sorted(ca_active - sa_active),
        "Both":     sorted(sa_active & ca_active),
        "Neither":  sorted((set(sa_ct) | set(ca_ct)) - sa_active - ca_active),
    }


print(f"Cell-type cascade classification (threshold = {ACTIVATION_THRESHOLD})")
print("SA-only  = direct targets (strong direct transcriptional signal)")
print("CA-only  = cascade responders (signal requires contextual reasoning)")
print("Both     = hub cells (active in both layers)")
print()

for cond in ORDERED_CONDITIONS:
    result = ct_results.get(cond)
    if result is None:
        continue
    classes = _classify_cell_types(result["sa"], result["ca"])
    difficulty = DIFFICULTY_MAP.get(cond, "")
    print(f"--- {cond}  ({difficulty}) ---")
    for label, cts in classes.items():
        print(f"  {label:<9}: {cts if cts else '(none)'}")
    print()

## 12. Normalized AUC vs P_max Scatter

In [ ]:
def _compute_normalized_auc(trajectory):
    """
    Normalized trajectory AUC: AUC(p_correct(t) / max(p_correct(t))).
    Captures learning shape independent of absolute confidence ceiling.
    """
    arr = np.array(trajectory)
    max_val = arr.max()
    if max_val < 1e-10:
        return 0.0
    normed = arr / max_val
    n = len(normed)
    return float(np.trapz(normed) / max(n - 1, 1))


def _aggregate_scatter_metrics(records, donor_traj):
    """
    For each condition: compute normalized AUC and final P per donor,
    then aggregate (median per donor, mean across donors).
    """
    # group records by (cytokine, donor)
    grouped = defaultdict(lambda: defaultdict(list))
    for rec in records:
        grouped[rec["cytokine"]][rec["donor"]].append(rec["p_correct_trajectory"])

    norm_auc_map = {}
    p_max_map    = {}

    for cond, donor_dict in grouped.items():
        donor_norm_aucs = []
        donor_p_maxs    = []
        for donor, trajs in donor_dict.items():
            # median across tubes per donor
            median_traj = np.median(trajs, axis=0)
            donor_norm_aucs.append(_compute_normalized_auc(median_traj))
            donor_p_maxs.append(float(median_traj[-1]))
        norm_auc_map[cond] = float(np.mean(donor_norm_aucs))
        p_max_map[cond]    = float(np.mean(donor_p_maxs))

    return norm_auc_map, p_max_map


fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (records_i, donor_traj_i, logged_i, stage_label) in zip(
    axes,
    [
        (records_s2, donor_traj_s2, logged_s2, "Stage 2 (SA)"),
        (records_s3, donor_traj_s3, logged_s3, "Stage 3 CA-only"),
    ],
):
    norm_auc_map, p_max_map = _aggregate_scatter_metrics(records_i, donor_traj_i)

    for cond in ORDERED_CONDITIONS:
        if cond not in norm_auc_map:
            continue
        x     = norm_auc_map[cond]
        y     = p_max_map[cond]
        color = COLOR_MAP.get(cond, "gray")
        ax.scatter(x, y, color=color, s=120, zorder=3)
        ax.annotate(
            f"{cond}\n({DIFFICULTY_MAP.get(cond, '')})",
            (x, y), textcoords="offset points", xytext=(6, 4), fontsize=8,
        )

    ax.set_xlabel("Normalized trajectory AUC  [shape metric]\n"
                  "AUC(p_correct(t) / max(p_correct(t))), donor-aggregated")
    ax.set_ylabel("Final P(Y_correct)  [ceiling metric]\n"
                  "p_correct(t_final), donor-aggregated")
    ax.set_title(f"{stage_label} — Oelen 2022")
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.axvline(0.5, color="gray", ls=":", lw=0.8)
    ax.axhline(0.5, color="gray", ls=":", lw=0.8)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    "Learnability scatter — Normalized AUC (shape) vs Final P (ceiling)\n"
    "Metric: AUC(norm_p_correct_trajectory) and p_correct(t_final), "
    "aggregated to donor level (median per donor, mean across donors)",
    fontsize=9,
)
plt.tight_layout()
plt.show()